# BritMart — Data Engineering Fundamentals Lab
**Week 2 · Day 1 | Questions 1–20**

This notebook contains explanations, tables, executable Pandas exercises, and a data-platform diagram. Run code cells from top to bottom.

## Part 1 — Data Engineering and Source Systems
### Q1 — Source inventory
| Source | Type of data | How often it may change | One check before using it |
|---|---|---|---|
| MySQL orders | Structured relational order records | Whenever orders are created or updated | Check order IDs are present and unique at the intended order grain |
| Supplier CSV | Structured tabular product data | When supplier sends a new file | Check required columns exist |
| Website JSON | Semi-structured website events | Continuously as users interact | Check required event fields are present |
| Excel stock report | Spreadsheet inventory data | When store updates or sends report | Check stock quantities are valid numbers |

### Q2 — Classify tasks
| Task | Main role | Why |
|---|---|---|
| A. Copy orders daily into analytical storage | Data engineering | Builds a repeatable ingestion pipeline |
| B. Chart sales by city | Data analytics | Summarizes data for interpretation |
| C. Predict customer churn | Data science | Trains a predictive model |
| D. Remove duplicate order records | Data engineering | Improves data quality before reporting |
| E. Rerun failed pipelines safely | Data engineering | Ensures reliable recovery |

### Q3 — Inspect three small input batches
Create three DataFrames, count rows, combine, and sum sales.

In [1]:
import pandas as pd

batch_1 = pd.DataFrame({"order_id": [101, 102], "city": ["London", "Manchester"], "amount_gbp": [45.50, 18.00]})
batch_2 = pd.DataFrame({"order_id": [103, 104], "city": ["Bristol", "London"], "amount_gbp": [62.25, 27.40]})
batch_3 = pd.DataFrame({"order_id": [105, 106], "city": ["Leeds", "Manchester"], "amount_gbp": [34.80, 51.10]})

print("Batch 1:", len(batch_1))
print("Batch 2:", len(batch_2))
print("Batch 3:", len(batch_3))

all_orders = pd.concat([batch_1, batch_2, batch_3], ignore_index=True)
print("Total rows:", len(all_orders))
print("Total sales:", round(all_orders["amount_gbp"].sum(), 2))

Batch 1: 2
Batch 2: 2
Batch 3: 2
Total rows: 6
Total sales: 239.05


**Expected:** 2 rows per batch, 6 total rows, **£239.05** total sales. `ignore_index=True` gives the combined table fresh row labels (0–5).

### Q4 — MCQ: Data engineering
**B. Building a reliable process that moves and cleans order data.** This is core data engineering work.

## Part 2 — Pipeline Lifecycle
### Q5 — Put the pipeline steps in order
1. Read new order records.
2. Store the received records.
3. Clean invalid values.
4. Create daily sales summary.
5. Make the summary available to analysts.
6. Check that the run completed.

**Note:** Monitoring should also happen throughout the pipeline, not only after the last step.

### Q6 — Design a daily-order pipeline
| Pipeline step | What happens? | Simple check |
|---|---|---|
| Source | Supplier sends nightly order CSV | Confirm expected file arrived |
| Ingestion | Read CSV into the pipeline | File opens and is not empty |
| Storage | Save an unchanged copy of received data | File stored and row count recorded |
| Processing | Validate amounts and IDs, clean values, calculate daily totals | Count valid and rejected rows; check totals |
| Serving | Publish daily order summary for analysts | Analysts can access latest summary |
| Monitoring | Track run status, row counts, errors, and freshness across stages | Alert on failures or unexpected counts |

### Q7 — Process three batches and keep counts
Add `batch_id`, combine, and group by batch.

In [2]:
batch_1["batch_id"] = "batch_1"
batch_2["batch_id"] = "batch_2"
batch_3["batch_id"] = "batch_3"

combined = pd.concat([batch_1, batch_2, batch_3], ignore_index=True)
summary = (combined.groupby("batch_id", as_index=False)
           .agg(row_count=("order_id", "size"), sales_gbp=("amount_gbp", "sum")))
summary["sales_gbp"] = summary["sales_gbp"].round(2)
print(summary.to_string(index=False))

batch_id  row_count  sales_gbp
 batch_1          2      63.50
 batch_2          2      89.65
 batch_3          2      85.90


**Expected summary:**

| batch_id | row_count | sales_gbp |
|---|---:|---:|
| batch_1 | 2 | 63.50 |
| batch_2 | 2 | 89.65 |
| batch_3 | 2 | 85.90 |

`as_index=False` keeps `batch_id` as a regular column.

### Q8 — MCQ: Ingestion
**A. Moving data from a source into the data platform.**

## Part 3 — Transformation, Batch and Streaming
### Q9 — Clean a batch of orders
Reject missing cities and non-positive amounts; trim spaces in valid city names.

In [3]:
orders = pd.DataFrame({
    "order_id": [201, 202, 203, 204],
    "city": [" London ", "Manchester", "Leeds", None],
    "amount_gbp": [45.50, -5.00, 30.00, 20.00],
})
invalid_mask = (orders["city"].isna() | (orders["amount_gbp"] <= 0))
rejected_orders = orders[invalid_mask].copy()
valid_orders = orders[~invalid_mask].copy()
valid_orders["city"] = valid_orders["city"].str.strip()
print("VALID ORDERS")
print(valid_orders.to_string(index=False))
print("\nREJECTED ORDERS")
print(rejected_orders.to_string(index=False))

VALID ORDERS
 order_id   city  amount_gbp
      201 London        45.5
      203  Leeds        30.0

REJECTED ORDERS
 order_id       city  amount_gbp
      202 Manchester        -5.0
      204       None        20.0


**Expected:** valid IDs **201, 203**; rejected IDs **202** (negative amount) and **204** (missing city). London has surrounding spaces removed.

### Q10 — Batch or streaming?
| Requirement | Choice | Reason |
|---|---|---|
| Daily supplier file at midnight | Batch | File arrives at a fixed time |
| Payment events needed within seconds | Streaming | Low-latency response required |
| Monthly finance report | Batch | Monthly scheduled processing is sufficient |
| Live website activity feed | Streaming | Events arrive continuously |

### Q11 — Simulate three batch runs
Track each batch’s count, total, and cumulative total.

In [4]:
batches = [[10, 20, 30], [40, 50], [60, 70, 80, 90]]
cumulative_total = 0
for batch_number, batch in enumerate(batches, start=1):
    batch_total = sum(batch)
    cumulative_total += batch_total
    print(f"Batch {batch_number}: record count={len(batch)}, batch total={batch_total}, cumulative total={cumulative_total}")

Batch 1: record count=3, batch total=60, cumulative total=60
Batch 2: record count=2, batch total=90, cumulative total=150
Batch 3: record count=4, batch total=300, cumulative total=450


**Expected:** Batch 1: count 3, total 60, cumulative 60; Batch 2: count 2, total 90, cumulative 150; Batch 3: count 4, total 300, cumulative 450.

### Q12 — MCQ: Streaming
**C. Fraud events that must be checked within seconds.**

## Part 4 — OLTP, OLAP, ETL, ELT and Platform Design
### Q13 — OLTP or OLAP?
| Workload | Type | Reason |
|---|---|---|
| A. Record a new customer order | OLTP | Records an operational transaction |
| B. Update one order’s payment status | OLTP | Updates an operational record |
| C. Sales by city over last 12 months | OLAP | Analyzes historical data |
| D. Average order value by product category | OLAP | Aggregates data for comparison |

### Q14 — ETL or ELT?
- **Design A = ETL:** Extract → Transform → Load. Data is cleaned before loading to the target.
- **Design B = ELT:** Extract → Load → Transform. Data is loaded first and transformed inside the target platform.

### Q15 — Small modern data platform
The diagram below has **four source systems**, ingestion, raw storage, processing, curated storage, serving, and monitoring.

```text
 MySQL Orders ────┐
 Supplier CSV ────┤
 Website JSON ────┼──► INGESTION ──► RAW STORAGE ──► PROCESSING ──► CURATED STORAGE ──► SERVING ──► REPORTS
 Excel Stock ─────┘                        (original)       (validate, clean)    (trusted tables)    (analysts)

 MONITORING: check ingestion, storage, processing, serving, row counts, freshness and failures throughout.
```

### Q16 — MCQ: OLAP
**D. Calculate three years of sales by region.**

## Optional Practice
### Q17 — Recover a failed batch
1. Check the pipeline logs and source file for **batch_2** to find why processing failed.
2. Verify **batch_1** was completed and recorded; do not process it again unnecessarily.
3. Fix the problem and rerun **batch_2** safely from its stored input.
4. Ensure the rerun does not duplicate previously written records (for example, replace the failed batch output or use a safe upsert).
5. Compare input, valid, rejected and output counts, and record batch status.
6. Start **batch_3** only when batch_2 is successfully reconciled.

### Q18 — Compare pipeline designs
**Design B: Source → Store received data → Clean → Final report** is better for reprocessing. Keeping the received data means the cleaning step can be rerun if business rules change, without needing to obtain the source data again.

### Q19 — Reconcile three batches
Difference = expected source count − processed count.

In [5]:
expected = {"batch_1": 100, "batch_2": 120, "batch_3": 80}
processed = {"batch_1": 100, "batch_2": 118, "batch_3": 80}
reconciliation = pd.DataFrame([
    {"batch_id": batch, "expected": count, "processed": processed[batch], "difference": count - processed[batch]}
    for batch, count in expected.items()
])
print(reconciliation.to_string(index=False))
print("Needs investigation:", reconciliation.loc[reconciliation["difference"] != 0, "batch_id"].tolist())

batch_id  expected  processed  difference
 batch_1       100        100           0
 batch_2       120        118           2
 batch_3        80         80           0
Needs investigation: ['batch_2']


**Expected differences:** batch_1 = 0, batch_2 = 2, batch_3 = 0. **Investigate batch_2**, which has two fewer processed records than expected. Some difference may be explained by documented valid rejections; otherwise it signals missing data.

### Q20 — Choose processing style
| Case | Choice | Reason |
|---|---|---|
| 1. Stock file every evening | Batch | Periodic file delivery |
| 2. Website click events all day | Streaming | Continuous events |
| 3. Finance report at month-end | Batch | Scheduled monthly output |
| 4. Payment alerts within seconds | Streaming | Low-latency alert requirement |
| 5. Supplier price file every Monday | Batch | Weekly file delivery |

## Final check
This workbook covers source systems, data-engineering responsibilities, ingestion, storage, processing, serving, monitoring, batch reconciliation, batch vs streaming, OLTP vs OLAP, ETL vs ELT, and a platform diagram.

**GitHub:** Upload this `.ipynb` file to a repository and share the repository link after checking the notebook renders correctly. No GitHub URL exists until you upload it.